<a href="https://colab.research.google.com/github/VasilisPapageorgiou/Amortization-of-Risk-Indicators/blob/main/Sim2_Bayesian_Embedding.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# SIMULATION 2
# High-dimensional heterogeneous allocation + computation
# Fixed-dimensional spectral shrinkage + blocked MCMC
#
# Google Colab: ONE CELL
#
# Objectives:
#   1. Heterogeneous dimension allocation across many predictors.
#   2. Predictive performance after adaptive budget selection.
#   3. Parameter reduction.
#   4. DP reduction of the candidate-refitting space.
#   5. Exact DP verification against exhaustive enumeration
#      on a manageable lower-dimensional subproblem.
#
# NO EJOR COMPARISON.
# ============================================================

# ---------------------------
# 0. START / RESUME QUESTION
# ---------------------------
MODE = input(
    "Simulation 2: type 1 to START FROM THE BEGINNING "
    "or 2 to RESUME FROM THE LAST CHECKPOINT: "
).strip()

if MODE not in {"1", "2"}:
    raise ValueError("Please run the cell again and enter either 1 or 2.")

# ---------------------------
# 1. IMPORTS
# ---------------------------
import os
import gc
import json
import math
import time
import pickle
import random
import hashlib
import itertools
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import spearmanr

import torch
import torch.nn as nn
import torch.nn.functional as F

warnings.filterwarnings("ignore")

DEVICE = torch.device("cpu")
torch.set_num_threads(max(1, min(4, os.cpu_count() or 1)))

# ---------------------------
# 2. GOOGLE DRIVE
# ---------------------------
try:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = "/content/drive/MyDrive/JRSSB_Embedding_Allocation/Simulation2"
except Exception:
    ROOT = "./JRSSB_Embedding_Allocation/Simulation2"
    print("Google Drive was not detected. Using local directory:", ROOT)

os.makedirs(ROOT, exist_ok=True)

CHECKPOINT_PATH = os.path.join(ROOT, "simulation2_checkpoint.pkl")
RESULTS_PATH = os.path.join(ROOT, "simulation2_replication_results.csv")
DIMENSION_PATH = os.path.join(ROOT, "simulation2_dimension_results.csv")
BUDGET_RESULTS_PATH = os.path.join(ROOT, "simulation2_budget_results.csv")
SUMMARY_PATH = os.path.join(ROOT, "simulation2_summary.csv")

# ---------------------------
# 3. CONFIGURATION
# ---------------------------

PILOT_MODE = True

BASE_SEED = 314159
CHECKPOINT_SECONDS = 20 * 60

# Many heterogeneous categorical predictors.
N_LEVELS = [
    3, 5, 8, 4, 10, 6,
    7, 12, 5, 9, 4, 11
]

M = len(N_LEVELS)

P_Z = 5
P_U = 3
Q = 1

K_LIST = [
    min(nj - 1, P_Z)
    for nj in N_LEVELS
]

N_TRAIN = 900
N_VAL = 300
N_TEST = 2000

ZETA_N = 0.010

THETA_LAMBDA0 = 0.15
TAU_GAMMA = 2.0
TAU_MU = 2.0
SIGMA_A0 = 2.5
SIGMA_B0 = 0.05

REF_HIDDEN = (28, 14)
RED_HIDDEN = (28, 14)

if PILOT_MODE:

    N_REPS = 6

    REF_EPOCHS = 260
    RED_EPOCHS = 210

    MCMC_ITERS = 550
    MCMC_BURN = 275
    MCMC_THIN = 3

else:

    N_REPS = 30

    REF_EPOCHS = 360
    RED_EPOCHS = 280

    MCMC_ITERS = 1100
    MCMC_BURN = 550
    MCMC_THIN = 4

REGIMES = [
    "heterogeneous",
    "near_homogeneous"
]

NOISE_SD = 0.25

MIN_COST = sum(N_LEVELS)

MAX_COST = sum(
    N_LEVELS[j] * K_LIST[j]
    for j in range(M)
)

BUDGETS = sorted(
    set(
        int(round(x))
        for x in np.linspace(
            MIN_COST,
            MAX_COST,
            7
        )
    )
)

BUDGETS[0] = MIN_COST
BUDGETS[-1] = MAX_COST
BUDGETS = sorted(set(BUDGETS))

# ---------------------------
# 4. CONFIG SIGNATURE
# ---------------------------

CONFIG_FOR_HASH = {
    "pilot": PILOT_MODE,
    "n_reps": N_REPS,
    "n_levels": N_LEVELS,
    "p_z": P_Z,
    "p_u": P_U,
    "n_train": N_TRAIN,
    "n_val": N_VAL,
    "n_test": N_TEST,
    "budgets": BUDGETS,
    "regimes": REGIMES,
    "noise": NOISE_SD,
    "zeta": ZETA_N,
    "mcmc_iters": MCMC_ITERS,
    "mcmc_burn": MCMC_BURN,
    "mcmc_thin": MCMC_THIN,
}

CONFIG_SIGNATURE = hashlib.md5(
    json.dumps(
        CONFIG_FOR_HASH,
        sort_keys=True
    ).encode()
).hexdigest()

# ---------------------------
# 5. REPRODUCIBILITY
# ---------------------------

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


# ---------------------------
# 6. EMBEDDING CONSTRUCTION
# ---------------------------

def centered_orthonormal_matrix(n, k, rng):

    A = rng.normal(
        size=(n, k)
    )

    A -= A.mean(
        axis=0,
        keepdims=True
    )

    Qm, _ = np.linalg.qr(A)

    Qm -= Qm.mean(
        axis=0,
        keepdims=True
    )

    Qm, _ = np.linalg.qr(Qm)

    return Qm[:, :k]


def make_theta_from_singular_values(
    n,
    pz,
    singular_values,
    rng
):
    k = min(n - 1, pz)

    U = centered_orthonormal_matrix(
        n,
        k,
        rng
    )

    A = rng.normal(
        size=(pz, k)
    )

    V, _ = np.linalg.qr(A)
    V = V[:, :k]

    s = np.asarray(
        singular_values[:k],
        dtype=float
    )

    theta = (
        U
        @ np.diag(s)
        @ V.T
    )

    theta -= theta.mean(
        axis=0,
        keepdims=True
    )

    return theta


# ---------------------------
# 7. HETEROGENEOUS SPECTRA
# ---------------------------

LOW = np.array([
    1.60, 0.22, 0.08, 0.03, 0.01
])

MEDIUM = np.array([
    1.60, 0.92, 0.34, 0.12, 0.04
])

HIGH = np.array([
    1.60, 1.30, 1.02, 0.74, 0.48
])

HOMOGENEOUS = np.array([
    1.55, 0.82, 0.31, 0.10, 0.03
])

COMPLEXITY_TYPES = [
    "low",
    "high",
    "medium",
    "low",
    "high",
    "medium",
    "high",
    "low",
    "medium",
    "high",
    "low",
    "high",
]


def get_spectra(regime):

    spectra = []

    labels = []

    for j in range(M):

        Kj = K_LIST[j]

        if regime == "heterogeneous":

            kind = COMPLEXITY_TYPES[j]

            if kind == "low":
                s = LOW
            elif kind == "medium":
                s = MEDIUM
            else:
                s = HIGH

            labels.append(kind)

        elif regime == "near_homogeneous":

            s = HOMOGENEOUS
            labels.append("common")

        else:
            raise ValueError(
                "Unknown regime."
            )

        spectra.append(
            np.asarray(
                s[:Kj],
                dtype=float
            )
        )

    return spectra, labels


# ---------------------------
# 8. TRUE NONLINEAR MAP
# ---------------------------

def make_teacher_weights(seed=90210):

    rng = np.random.default_rng(seed)

    h1, h2 = REF_HIDDEN

    W1 = rng.normal(
        scale=0.65 / np.sqrt(P_Z),
        size=(h1, P_Z)
    )

    b1 = rng.normal(
        scale=0.10,
        size=h1
    )

    W2 = rng.normal(
        scale=0.65 / np.sqrt(h1),
        size=(h2, h1)
    )

    b2 = rng.normal(
        scale=0.10,
        size=h2
    )

    W3 = rng.normal(
        scale=0.90 / np.sqrt(h2),
        size=(1, h2)
    )

    b3 = np.array([0.05])

    return [
        (W1, b1),
        (W2, b2),
        (W3, b3)
    ]


TEACHER_WEIGHTS = make_teacher_weights()


def mlp_forward_numpy(z, weights):

    h = np.asarray(
        z,
        dtype=float
    )

    for ell, (W, b) in enumerate(weights):

        h = (
            h @ W.T
            + b
        )

        if ell < len(weights) - 1:
            h = np.tanh(h)

    return h


# ---------------------------
# 9. TRUE PARAMETERS
# ---------------------------

def build_true_parameters(regime):

    rng = np.random.default_rng(4567)

    spectra, labels = get_spectra(
        regime
    )

    theta = []

    for j in range(M):

        theta.append(
            make_theta_from_singular_values(
                N_LEVELS[j],
                P_Z,
                spectra[j],
                rng
            )
        )

    gamma = rng.normal(
        scale=0.28,
        size=(P_U, P_Z)
    )

    mu = np.array([
        0.12,
        -0.08,
        0.05,
        0.07,
        -0.04
    ])

    return {
        "theta": theta,
        "gamma": gamma,
        "mu": mu,
        "spectra": spectra,
        "labels": labels,
    }


# ---------------------------
# 10. DATA GENERATION
# ---------------------------

def generate_dataset(
    n,
    true_par,
    noise_sd,
    seed
):

    rng = np.random.default_rng(seed)

    C = [
        rng.integers(
            0,
            N_LEVELS[j],
            size=n
        )
        for j in range(M)
    ]

    U = rng.normal(
        size=(n, P_U)
    )

    z = np.tile(
        true_par["mu"],
        (n, 1)
    )

    z += U @ true_par["gamma"]

    for j in range(M):

        z += true_par["theta"][j][
            C[j],
            :
        ]

    mean = mlp_forward_numpy(
        z,
        TEACHER_WEIGHTS
    )

    y = (
        mean
        + noise_sd
        * rng.normal(
            size=mean.shape
        )
    )

    return C, U, y


# ---------------------------
# 11. REFERENCE MODEL
# ---------------------------

class ReferenceModel(nn.Module):

    def __init__(self):
        super().__init__()

        self.embeddings = nn.ModuleList(
            [
                nn.Embedding(
                    N_LEVELS[j],
                    P_Z
                )
                for j in range(M)
            ]
        )

        for emb in self.embeddings:
            nn.init.normal_(
                emb.weight,
                std=0.10
            )

        self.gamma = nn.Parameter(
            0.10
            * torch.randn(
                P_U,
                P_Z
            )
        )

        self.mu = nn.Parameter(
            torch.zeros(P_Z)
        )

        self.mlp = nn.Sequential(
            nn.Linear(
                P_Z,
                REF_HIDDEN[0]
            ),
            nn.Tanh(),
            nn.Linear(
                REF_HIDDEN[0],
                REF_HIDDEN[1]
            ),
            nn.Tanh(),
            nn.Linear(
                REF_HIDDEN[1],
                1
            )
        )

    def forward(self, C, U):

        z = (
            self.mu.unsqueeze(0)
            + U @ self.gamma
        )

        for j in range(M):

            z = (
                z
                + self.embeddings[j](
                    C[j]
                )
            )

        return self.mlp(z)

    @torch.no_grad()
    def center_embeddings_(self):

        for emb in self.embeddings:

            emb.weight -= (
                emb.weight.mean(
                    dim=0,
                    keepdim=True
                )
            )


def train_reference_model(
    C,
    U,
    y,
    seed
):

    set_seed(seed)

    model = ReferenceModel().to(
        DEVICE
    )

    C_t = [
        torch.tensor(
            x,
            dtype=torch.long,
            device=DEVICE
        )
        for x in C
    ]

    U_t = torch.tensor(
        U,
        dtype=torch.float32,
        device=DEVICE
    )

    y_t = torch.tensor(
        y,
        dtype=torch.float32,
        device=DEVICE
    )

    opt = torch.optim.Adam(
        model.parameters(),
        lr=0.010,
        weight_decay=1e-5
    )

    for _ in range(REF_EPOCHS):

        opt.zero_grad()

        pred = model(
            C_t,
            U_t
        )

        loss = F.mse_loss(
            pred,
            y_t
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            5.0
        )

        opt.step()
        model.center_embeddings_()

    return model


@torch.no_grad()
def predict_reference(
    model,
    C,
    U
):

    C_t = [
        torch.tensor(
            x,
            dtype=torch.long,
            device=DEVICE
        )
        for x in C
    ]

    U_t = torch.tensor(
        U,
        dtype=torch.float32,
        device=DEVICE
    )

    return (
        model(C_t, U_t)
        .cpu()
        .numpy()
    )


def extract_reference_parameters(
    model
):

    theta = [
        emb.weight.detach()
        .cpu()
        .numpy()
        .astype(float)
        .copy()
        for emb in model.embeddings
    ]

    theta = [
        x - x.mean(
            axis=0,
            keepdims=True
        )
        for x in theta
    ]

    gamma = (
        model.gamma.detach()
        .cpu()
        .numpy()
        .astype(float)
        .copy()
    )

    mu = (
        model.mu.detach()
        .cpu()
        .numpy()
        .astype(float)
        .copy()
    )

    weights = []

    for layer in model.mlp:

        if isinstance(
            layer,
            nn.Linear
        ):

            weights.append(
                (
                    layer.weight.detach()
                    .cpu()
                    .numpy()
                    .astype(float)
                    .copy(),

                    layer.bias.detach()
                    .cpu()
                    .numpy()
                    .astype(float)
                    .copy()
                )
            )

    return theta, gamma, mu, weights


# ---------------------------
# 12. PRIORS
# ---------------------------

def log_theta_prior(theta):

    k = min(
        theta.shape[0] - 1,
        theta.shape[1]
    )

    s = np.linalg.svd(
        theta,
        compute_uv=False
    )[:k]

    lam = (
        THETA_LAMBDA0
        * np.arange(
            1,
            k + 1
        ) ** 2
    )

    return -0.5 * np.sum(
        lam * s ** 2
    )


def log_gamma_mu_prior(
    gamma,
    mu
):

    return (
        -0.5
        * np.sum(gamma ** 2)
        / TAU_GAMMA ** 2
        -
        0.5
        * np.sum(mu ** 2)
        / TAU_MU ** 2
    )


# ---------------------------
# 13. LIKELIHOOD
# ---------------------------

def gaussian_loglik(
    y,
    yhat,
    sigma2
):

    residual = y - yhat

    return (
        -0.5
        * y.size
        * np.log(
            2.0
            * np.pi
            * sigma2
        )
        -
        0.5
        * np.sum(
            residual ** 2
        )
        / sigma2
    )


def latent_numpy(
    C,
    U,
    theta,
    gamma,
    mu
):

    z = np.tile(
        mu,
        (len(C[0]), 1)
    )

    z += U @ gamma

    for j in range(M):

        z += theta[j][
            C[j],
            :
        ]

    return z


# ---------------------------
# 14. BLOCKED MCMC
# ---------------------------

def blocked_mcmc(
    C,
    U,
    y,
    theta_init,
    gamma_init,
    mu_init,
    fixed_weights,
    sigma2_init,
    seed,
    heartbeat=None
):

    rng = np.random.default_rng(
        seed
    )

    theta = [
        x.copy()
        for x in theta_init
    ]

    gamma = gamma_init.copy()
    mu = mu_init.copy()

    sigma2 = float(
        max(
            sigma2_init,
            1e-5
        )
    )

    z_cur = latent_numpy(
        C,
        U,
        theta,
        gamma,
        mu
    )

    yhat_cur = mlp_forward_numpy(
        z_cur,
        fixed_weights
    )

    ll_cur = gaussian_loglik(
        y,
        yhat_cur,
        sigma2
    )

    theta_prior = [
        log_theta_prior(x)
        for x in theta
    ]

    gm_prior = log_gamma_mu_prior(
        gamma,
        mu
    )

    theta_steps = np.full(
        M,
        0.022
    )

    gm_step = 0.018

    theta_accept = np.zeros(M)
    theta_trials = np.zeros(M)

    win_accept = np.zeros(M)
    win_trials = np.zeros(M)

    gm_accept = 0
    gm_trials = 0

    gm_win_accept = 0
    gm_win_trials = 0

    singular_draws = [
        []
        for _ in range(M)
    ]

    target = 0.25
    adapt_every = 50

    for it in range(MCMC_ITERS):

        # Predictor-wise Theta blocks.
        for j in range(M):

            proposal = (
                theta[j]
                + theta_steps[j]
                * rng.normal(
                    size=theta[j].shape
                )
            )

            proposal -= proposal.mean(
                axis=0,
                keepdims=True
            )

            z_prop = (
                z_cur
                + proposal[
                    C[j],
                    :
                ]
                - theta[j][
                    C[j],
                    :
                ]
            )

            yhat_prop = mlp_forward_numpy(
                z_prop,
                fixed_weights
            )

            ll_prop = gaussian_loglik(
                y,
                yhat_prop,
                sigma2
            )

            prior_prop = log_theta_prior(
                proposal
            )

            log_ratio = (
                ll_prop
                + prior_prop
                - ll_cur
                - theta_prior[j]
            )

            theta_trials[j] += 1
            win_trials[j] += 1

            if np.log(
                rng.uniform()
            ) < log_ratio:

                theta[j] = proposal
                z_cur = z_prop
                yhat_cur = yhat_prop
                ll_cur = ll_prop
                theta_prior[j] = prior_prop

                theta_accept[j] += 1
                win_accept[j] += 1

        # Joint Gamma and mu.
        gamma_prop = (
            gamma
            + gm_step
            * rng.normal(
                size=gamma.shape
            )
        )

        mu_prop = (
            mu
            + gm_step
            * rng.normal(
                size=mu.shape
            )
        )

        z_prop = (
            z_cur
            + U @ (
                gamma_prop
                - gamma
            )
            + (
                mu_prop
                - mu
            )
        )

        yhat_prop = mlp_forward_numpy(
            z_prop,
            fixed_weights
        )

        ll_prop = gaussian_loglik(
            y,
            yhat_prop,
            sigma2
        )

        prior_prop = log_gamma_mu_prior(
            gamma_prop,
            mu_prop
        )

        log_ratio = (
            ll_prop
            + prior_prop
            - ll_cur
            - gm_prior
        )

        gm_trials += 1
        gm_win_trials += 1

        if np.log(
            rng.uniform()
        ) < log_ratio:

            gamma = gamma_prop
            mu = mu_prop

            z_cur = z_prop
            yhat_cur = yhat_prop

            ll_cur = ll_prop
            gm_prior = prior_prop

            gm_accept += 1
            gm_win_accept += 1

        # Gibbs sigma^2.
        resid = y - yhat_cur

        shape = (
            SIGMA_A0
            + 0.5 * y.size
        )

        rate = (
            SIGMA_B0
            + 0.5
            * np.sum(
                resid ** 2
            )
        )

        sigma2 = 1.0 / rng.gamma(
            shape=shape,
            scale=1.0 / rate
        )

        ll_cur = gaussian_loglik(
            y,
            yhat_cur,
            sigma2
        )

        # Proposal adaptation.
        if (
            it < MCMC_BURN
            and
            (it + 1) % adapt_every == 0
        ):

            for j in range(M):

                if win_trials[j] > 0:

                    ar = (
                        win_accept[j]
                        / win_trials[j]
                    )

                    theta_steps[j] *= np.exp(
                        0.45
                        * (ar - target)
                    )

                    theta_steps[j] = np.clip(
                        theta_steps[j],
                        0.001,
                        0.12
                    )

            if gm_win_trials > 0:

                ar = (
                    gm_win_accept
                    / gm_win_trials
                )

                gm_step *= np.exp(
                    0.45
                    * (ar - target)
                )

                gm_step = float(
                    np.clip(
                        gm_step,
                        0.001,
                        0.12
                    )
                )

            win_accept[:] = 0
            win_trials[:] = 0

            gm_win_accept = 0
            gm_win_trials = 0

        # Store only quantities required by S(d).
        if (
            it >= MCMC_BURN
            and
            (it - MCMC_BURN)
            % MCMC_THIN
            == 0
        ):

            for j in range(M):

                singular_draws[j].append(
                    np.linalg.svd(
                        theta[j],
                        compute_uv=False
                    )[:K_LIST[j]]
                )

        if (
            heartbeat is not None
            and (it + 1) % 50 == 0
        ):
            heartbeat()

    singular_draws = [
        np.asarray(x)
        for x in singular_draws
    ]

    mean_s2 = [
        np.mean(
            x ** 2,
            axis=0
        )
        for x in singular_draws
    ]

    return {
        "mean_s2": mean_s2,
        "singular_draws": singular_draws,
        "theta_acceptance": (
            theta_accept
            / np.maximum(
                theta_trials,
                1
            )
        ),
        "gm_acceptance": (
            gm_accept
            / max(
                gm_trials,
                1
            )
        ),
    }


# ---------------------------
# 15. STRUCTURAL LOSSES
# ---------------------------

def tail_loss_arrays(
    mean_s2
):

    out = []

    for s2 in mean_s2:

        k = len(s2)

        out.append(
            np.array(
                [
                    np.sum(
                        s2[d:]
                    )
                    for d in range(
                        1,
                        k + 1
                    )
                ]
            )
        )

    return out


def allocation_loss(
    d,
    loss_arrays
):

    return float(
        sum(
            loss_arrays[j][
                d[j] - 1
            ]
            for j in range(
                len(d)
            )
        )
    )


def embedding_cost(d):

    return int(
        sum(
            N_LEVELS[j]
            * d[j]
            for j in range(M)
        )
    )


# ---------------------------
# 16. MCKP DYNAMIC PROGRAMMING
# ---------------------------

def dp_allocations(
    n_levels,
    k_list,
    loss_arrays,
    budgets
):

    m = len(n_levels)
    bmax = max(budgets)

    dp = np.full(
        (m + 1, bmax + 1),
        np.inf
    )

    parent_d = np.full(
        (m + 1, bmax + 1),
        -1,
        dtype=int
    )

    parent_b = np.full(
        (m + 1, bmax + 1),
        -1,
        dtype=int
    )

    dp[0, 0] = 0.0

    for j in range(1, m + 1):

        Nj = n_levels[j - 1]
        Kj = k_list[j - 1]

        for b in range(
            bmax + 1
        ):

            for d in range(
                1,
                Kj + 1
            ):

                cost = Nj * d

                if (
                    b >= cost
                    and
                    np.isfinite(
                        dp[
                            j - 1,
                            b - cost
                        ]
                    )
                ):

                    candidate = (
                        dp[
                            j - 1,
                            b - cost
                        ]
                        +
                        loss_arrays[
                            j - 1
                        ][
                            d - 1
                        ]
                    )

                    if candidate < dp[j, b]:

                        dp[j, b] = candidate
                        parent_d[j, b] = d
                        parent_b[j, b] = (
                            b - cost
                        )

    result = {}

    for B in budgets:

        feasible = np.arange(
            B + 1
        )

        values = dp[
            m,
            feasible
        ]

        cost_star = int(
            feasible[
                np.nanargmin(
                    values
                )
            ]
        )

        dvec = [None] * m
        b = cost_star

        for j in range(
            m,
            0,
            -1
        ):

            d = int(
                parent_d[j, b]
            )

            dvec[j - 1] = d

            b = int(
                parent_b[j, b]
            )

        result[B] = {
            "d": tuple(dvec),
            "loss": float(
                dp[
                    m,
                    cost_star
                ]
            ),
            "actual_cost": (
                cost_star
            )
        }

    return result


# ---------------------------
# 17. REDUCED MODEL
# ---------------------------

class ReducedModel(nn.Module):

    def __init__(self, d):
        super().__init__()

        self.d = list(d)

        self.embeddings = nn.ModuleList(
            [
                nn.Embedding(
                    N_LEVELS[j],
                    self.d[j]
                )
                for j in range(M)
            ]
        )

        for emb in self.embeddings:
            nn.init.normal_(
                emb.weight,
                std=0.10
            )

        self.gamma = nn.Parameter(
            0.10
            * torch.randn(
                P_U,
                P_Z
            )
        )

        self.mu = nn.Parameter(
            torch.zeros(P_Z)
        )

        input_dim = (
            sum(self.d)
            + P_Z
        )

        self.mlp = nn.Sequential(
            nn.Linear(
                input_dim,
                RED_HIDDEN[0]
            ),
            nn.Tanh(),
            nn.Linear(
                RED_HIDDEN[0],
                RED_HIDDEN[1]
            ),
            nn.Tanh(),
            nn.Linear(
                RED_HIDDEN[1],
                1
            )
        )

    def forward(self, C, U):

        pieces = [
            self.embeddings[j](
                C[j]
            )
            for j in range(M)
        ]

        continuous = (
            self.mu.unsqueeze(0)
            + U @ self.gamma
        )

        pieces.append(
            continuous
        )

        x = torch.cat(
            pieces,
            dim=1
        )

        return self.mlp(x)

    @torch.no_grad()
    def center_embeddings_(self):

        for emb in self.embeddings:

            emb.weight -= (
                emb.weight.mean(
                    dim=0,
                    keepdim=True
                )
            )


def train_reduced_model(
    C,
    U,
    y,
    d,
    seed
):

    set_seed(seed)

    model = ReducedModel(
        d
    ).to(DEVICE)

    C_t = [
        torch.tensor(
            x,
            dtype=torch.long,
            device=DEVICE
        )
        for x in C
    ]

    U_t = torch.tensor(
        U,
        dtype=torch.float32,
        device=DEVICE
    )

    y_t = torch.tensor(
        y,
        dtype=torch.float32,
        device=DEVICE
    )

    opt = torch.optim.Adam(
        model.parameters(),
        lr=0.010,
        weight_decay=1e-5
    )

    for _ in range(
        RED_EPOCHS
    ):

        opt.zero_grad()

        pred = model(
            C_t,
            U_t
        )

        loss = F.mse_loss(
            pred,
            y_t
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            5.0
        )

        opt.step()
        model.center_embeddings_()

    return model


@torch.no_grad()
def predict_reduced(
    model,
    C,
    U
):

    C_t = [
        torch.tensor(
            x,
            dtype=torch.long,
            device=DEVICE
        )
        for x in C
    ]

    U_t = torch.tensor(
        U,
        dtype=torch.float32,
        device=DEVICE
    )

    return (
        model(C_t, U_t)
        .cpu()
        .numpy()
    )


def mse(y, pred):
    return float(
        np.mean(
            (y - pred) ** 2
        )
    )


# ---------------------------
# 18. COMMON-DIMENSION BASELINE
# ---------------------------

def common_dimension_allocation(
    B
):
    best = tuple(
        1
        for _ in range(M)
    )

    for d in range(
        1,
        max(K_LIST) + 1
    ):

        candidate = tuple(
            min(
                d,
                K_LIST[j]
            )
            for j in range(M)
        )

        if embedding_cost(
            candidate
        ) <= B:

            best = candidate

        else:
            break

    return best


# ---------------------------
# 19. TRUE EFFECTIVE DIMENSION
# ---------------------------

def effective_dimension_90(
    singular_values
):

    s2 = (
        np.asarray(
            singular_values
        ) ** 2
    )

    total = np.sum(s2)

    cumulative = np.cumsum(
        s2
    )

    return int(
        np.searchsorted(
            cumulative,
            0.90 * total
        )
        + 1
    )


# ---------------------------
# 20. EXHAUSTIVE CHECK ON PREFIX
# ---------------------------

def verify_dp_by_enumeration(
    loss_arrays,
    n_prefix=6
):

    n_sub = N_LEVELS[
        :n_prefix
    ]

    k_sub = K_LIST[
        :n_prefix
    ]

    loss_sub = loss_arrays[
        :n_prefix
    ]

    min_b = sum(n_sub)

    max_b = sum(
        n_sub[j]
        * k_sub[j]
        for j in range(
            n_prefix
        )
    )

    B = int(
        round(
            0.60 * max_b
            + 0.40 * min_b
        )
    )

    t0 = time.perf_counter()

    dp_result = dp_allocations(
        n_sub,
        k_sub,
        loss_sub,
        [B]
    )[B]

    dp_time = (
        time.perf_counter()
        - t0
    )

    t1 = time.perf_counter()

    best_loss = np.inf
    best_d = None

    for d in itertools.product(
        *[
            range(
                1,
                k_sub[j] + 1
            )
            for j in range(
                n_prefix
            )
        ]
    ):

        cost = sum(
            n_sub[j] * d[j]
            for j in range(
                n_prefix
            )
        )

        if cost <= B:

            loss = float(
                sum(
                    loss_sub[j][
                        d[j] - 1
                    ]
                    for j in range(
                        n_prefix
                    )
                )
            )

            if loss < best_loss:
                best_loss = loss
                best_d = d

    exhaustive_time = (
        time.perf_counter()
        - t1
    )

    exact = bool(
        np.isclose(
            dp_result["loss"],
            best_loss,
            atol=1e-10,
            rtol=1e-10
        )
    )

    return {
        "exact": exact,
        "dp_time": dp_time,
        "exhaustive_time": exhaustive_time,
        "speedup": (
            exhaustive_time
            / max(
                dp_time,
                1e-12
            )
        ),
        "dp_d": dp_result["d"],
        "enum_d": best_d,
    }


# ---------------------------
# 21. CHECKPOINT STATE
# ---------------------------

if (
    MODE == "2"
    and
    os.path.exists(
        CHECKPOINT_PATH
    )
):

    with open(
        CHECKPOINT_PATH,
        "rb"
    ) as f:

        STATE = pickle.load(f)

    if (
        STATE.get(
            "config_signature"
        )
        != CONFIG_SIGNATURE
    ):

        raise RuntimeError(
            "Checkpoint configuration does not "
            "match the current simulation."
        )

    print(
        "Resumed checkpoint with",
        len(
            STATE["completed"]
        ),
        "completed replications."
    )

else:

    STATE = {
        "config_signature": (
            CONFIG_SIGNATURE
        ),
        "completed": set(),
        "replication_rows": [],
        "budget_rows": [],
        "dimension_rows": [],
    }

    if MODE == "2":
        print(
            "Checkpoint not found. "
            "Starting from the beginning."
        )


CHECKPOINT_CLOCK = {
    "last": time.time()
}


def save_checkpoint(
    force=False
):

    now = time.time()

    if (
        force
        or
        now
        - CHECKPOINT_CLOCK["last"]
        >= CHECKPOINT_SECONDS
    ):

        tmp = (
            CHECKPOINT_PATH
            + ".tmp"
        )

        with open(
            tmp,
            "wb"
        ) as f:

            pickle.dump(
                STATE,
                f,
                protocol=pickle.HIGHEST_PROTOCOL
            )

        os.replace(
            tmp,
            CHECKPOINT_PATH
        )

        CHECKPOINT_CLOCK[
            "last"
        ] = now

        print(
            "[checkpoint]",
            time.strftime(
                "%H:%M:%S"
            )
        )


# ---------------------------
# 22. ONE REPLICATION
# ---------------------------

def run_replication(
    regime,
    regime_index,
    rep
):

    t_start = time.time()

    seed = (
        BASE_SEED
        + 100000
        * regime_index
        + rep
    )

    true_par = (
        build_true_parameters(
            regime
        )
    )

    Ctr, Utr, ytr = generate_dataset(
        N_TRAIN,
        true_par,
        NOISE_SD,
        seed + 11
    )

    Cva, Uva, yva = generate_dataset(
        N_VAL,
        true_par,
        NOISE_SD,
        seed + 22
    )

    Cte, Ute, yte = generate_dataset(
        N_TEST,
        true_par,
        NOISE_SD,
        seed + 33
    )

    # ---------------------------------
    # Rich reference model
    # ---------------------------------
    reference = train_reference_model(
        Ctr,
        Utr,
        ytr,
        seed + 100
    )

    train_pred = predict_reference(
        reference,
        Ctr,
        Utr
    )

    test_pred_ref = predict_reference(
        reference,
        Cte,
        Ute
    )

    reference_test_mse = mse(
        yte,
        test_pred_ref
    )

    theta0, gamma0, mu0, omega_fixed = (
        extract_reference_parameters(
            reference
        )
    )

    sigma2_init = max(
        mse(
            ytr,
            train_pred
        ),
        1e-5
    )

    # ---------------------------------
    # Posterior inference
    # ---------------------------------
    posterior = blocked_mcmc(
        Ctr,
        Utr,
        ytr,
        theta0,
        gamma0,
        mu0,
        omega_fixed,
        sigma2_init,
        seed + 200,
        heartbeat=lambda: save_checkpoint(
            force=False
        )
    )

    posterior_losses = tail_loss_arrays(
        posterior["mean_s2"]
    )

    # ---------------------------------
    # DP
    # ---------------------------------
    t_dp = time.perf_counter()

    dp_results = dp_allocations(
        N_LEVELS,
        K_LIST,
        posterior_losses,
        BUDGETS
    )

    dp_runtime = (
        time.perf_counter()
        - t_dp
    )

    unique_allocations = sorted(
        set(
            dp_results[B]["d"]
            for B in BUDGETS
        )
    )

    total_allocation_space = int(
        np.prod(
            K_LIST,
            dtype=np.int64
        )
    )

    refit_fraction = (
        len(unique_allocations)
        / total_allocation_space
    )

    # Exact DP verification on a prefix.
    dp_check = verify_dp_by_enumeration(
        posterior_losses,
        n_prefix=6
    )

    # ---------------------------------
    # Refit structural candidates
    # ---------------------------------
    candidate_cache = {}

    for idx, d in enumerate(
        unique_allocations
    ):

        model = train_reduced_model(
            Ctr,
            Utr,
            ytr,
            d,
            seed
            + 10000
            + 100 * idx
        )

        val_pred = predict_reduced(
            model,
            Cva,
            Uva
        )

        test_pred = predict_reduced(
            model,
            Cte,
            Ute
        )

        candidate_cache[d] = {
            "val_mse": mse(
                yva,
                val_pred
            ),
            "test_mse": mse(
                yte,
                test_pred
            ),
            "total_parameters": int(
                sum(
                    p.numel()
                    for p in model.parameters()
                )
            )
        }

        del model
        gc.collect()

    # ---------------------------------
    # Adaptive budget
    # ---------------------------------
    BMAX = max(BUDGETS)

    d_max = dp_results[
        BMAX
    ]["d"]

    max_val = candidate_cache[
        d_max
    ]["val_mse"]

    admissible = [
        B
        for B in sorted(
            BUDGETS
        )
        if (
            candidate_cache[
                dp_results[B]["d"]
            ]["val_mse"]
            <= max_val + ZETA_N
        )
    ]

    B_hat = min(
        admissible
    )

    d_hat = dp_results[
        B_hat
    ]["d"]

    final_candidate = candidate_cache[
        d_hat
    ]

    max_candidate = candidate_cache[
        d_max
    ]

    # ---------------------------------
    # Common-dimension comparator
    # ---------------------------------
    d_common = (
        common_dimension_allocation(
            B_hat
        )
    )

    common_model = train_reduced_model(
        Ctr,
        Utr,
        ytr,
        d_common,
        seed + 55555
    )

    common_val = mse(
        yva,
        predict_reduced(
            common_model,
            Cva,
            Uva
        )
    )

    common_test = mse(
        yte,
        predict_reduced(
            common_model,
            Cte,
            Ute
        )
    )

    common_total_params = int(
        sum(
            p.numel()
            for p in common_model.parameters()
        )
    )

    del common_model
    gc.collect()

    # ---------------------------------
    # True spectral complexity
    # ---------------------------------
    d90 = [
        effective_dimension_90(
            true_par["spectra"][j]
        )
        for j in range(M)
    ]

    if (
        len(
            np.unique(d90)
        ) > 1
        and
        len(
            np.unique(d_hat)
        ) > 1
    ):

        complexity_corr = float(
            spearmanr(
                d90,
                d_hat
            ).correlation
        )

    else:
        complexity_corr = np.nan

    # ---------------------------------
    # Oracle structural allocation
    # ---------------------------------
    true_s2 = [
        np.asarray(s) ** 2
        for s in true_par["spectra"]
    ]

    true_losses = tail_loss_arrays(
        true_s2
    )

    oracle_at_selected_B = (
        dp_allocations(
            N_LEVELS,
            K_LIST,
            true_losses,
            [B_hat]
        )[B_hat]
    )

    true_structural_loss_selected = (
        allocation_loss(
            d_hat,
            true_losses
        )
    )

    true_structural_loss_oracle = (
        oracle_at_selected_B[
            "loss"
        ]
    )

    structural_excess = (
        true_structural_loss_selected
        -
        true_structural_loss_oracle
    )

    # ---------------------------------
    # Parameter counts
    # ---------------------------------
    rich_embedding_params = sum(
        N_LEVELS[j] * P_Z
        for j in range(M)
    )

    adaptive_embedding_params = (
        embedding_cost(
            d_hat
        )
    )

    common_embedding_params = (
        embedding_cost(
            d_common
        )
    )

    embedding_reduction = (
        1
        -
        adaptive_embedding_params
        / rich_embedding_params
    )

    # ---------------------------------
    # Budget records
    # ---------------------------------
    budget_rows = []

    for B in BUDGETS:

        d = dp_results[B]["d"]

        budget_rows.append(
            {
                "regime": regime,
                "rep": rep,
                "B": B,
                "d": str(tuple(d)),
                "actual_cost": (
                    embedding_cost(d)
                ),
                "structural_score": (
                    dp_results[B]["loss"]
                ),
                "validation_mse": (
                    candidate_cache[d][
                        "val_mse"
                    ]
                ),
                "test_mse": (
                    candidate_cache[d][
                        "test_mse"
                    ]
                ),
            }
        )

    # ---------------------------------
    # Predictor-level records
    # ---------------------------------
    dimension_rows = []

    for j in range(M):

        dimension_rows.append(
            {
                "regime": regime,
                "rep": rep,
                "predictor": j + 1,
                "N_j": N_LEVELS[j],
                "K_j": K_LIST[j],
                "complexity_type": (
                    true_par[
                        "labels"
                    ][j]
                ),
                "true_effective_dimension_90": (
                    d90[j]
                ),
                "selected_dimension": (
                    d_hat[j]
                ),
            }
        )

    # ---------------------------------
    # Replication summary
    # ---------------------------------
    row = {
        "regime": regime,
        "rep": rep,
        "selected_budget": B_hat,
        "selected_d": str(
            tuple(d_hat)
        ),
        "common_d": str(
            tuple(d_common)
        ),
        "adaptive_test_mse": (
            final_candidate[
                "test_mse"
            ]
        ),
        "common_dimension_test_mse": (
            common_test
        ),
        "common_dimension_validation_mse": (
            common_val
        ),
        "max_budget_test_mse": (
            max_candidate[
                "test_mse"
            ]
        ),
        "reference_full_test_mse": (
            reference_test_mse
        ),
        "adaptive_vs_common_test_difference": (
            final_candidate[
                "test_mse"
            ]
            - common_test
        ),
        "adaptive_vs_max_budget_test_difference": (
            final_candidate[
                "test_mse"
            ]
            - max_candidate[
                "test_mse"
            ]
        ),
        "adaptive_embedding_parameters": (
            adaptive_embedding_params
        ),
        "common_embedding_parameters": (
            common_embedding_params
        ),
        "rich_embedding_parameters": (
            rich_embedding_params
        ),
        "embedding_parameter_reduction": (
            embedding_reduction
        ),
        "adaptive_total_parameters": (
            final_candidate[
                "total_parameters"
            ]
        ),
        "common_total_parameters": (
            common_total_params
        ),
        "true_structural_excess_at_selected_budget": (
            structural_excess
        ),
        "dimension_complexity_spearman": (
            complexity_corr
        ),
        "total_allocation_space": (
            total_allocation_space
        ),
        "number_unique_refits": (
            len(
                unique_allocations
            )
        ),
        "refit_fraction_of_full_space": (
            refit_fraction
        ),
        "refit_reduction_percent": (
            100
            * (
                1
                - refit_fraction
            )
        ),
        "dp_runtime_seconds": (
            dp_runtime
        ),
        "dp_exact_on_prefix": (
            float(
                dp_check[
                    "exact"
                ]
            )
        ),
        "prefix_dp_time_seconds": (
            dp_check[
                "dp_time"
            ]
        ),
        "prefix_exhaustive_time_seconds": (
            dp_check[
                "exhaustive_time"
            ]
        ),
        "prefix_dp_speedup": (
            dp_check[
                "speedup"
            ]
        ),
        "mean_theta_acceptance": float(
            np.mean(
                posterior[
                    "theta_acceptance"
                ]
            )
        ),
        "gamma_mu_acceptance": float(
            posterior[
                "gm_acceptance"
            ]
        ),
        "runtime_seconds": (
            time.time()
            - t_start
        ),
    }

    del reference
    gc.collect()

    return (
        row,
        budget_rows,
        dimension_rows
    )


# ---------------------------
# 23. RUN
# ---------------------------

print("\nSimulation 2")
print("---------------------------------------")
print("Number of categorical predictors:", M)
print("N_j:", N_LEVELS)
print("K_j:", K_LIST)
print("Candidate budgets:", BUDGETS)
print(
    "Total possible allocations:",
    int(
        np.prod(
            K_LIST,
            dtype=np.int64
        )
    )
)
print("Pilot mode:", PILOT_MODE)
print("---------------------------------------\n")

try:

    for r_idx, regime in enumerate(
        REGIMES
    ):

        for rep in range(
            1,
            N_REPS + 1
        ):

            task = (
                regime,
                rep
            )

            if task in STATE[
                "completed"
            ]:
                continue

            print(
                f"\n[{regime}] "
                f"replication {rep}/{N_REPS}"
            )

            (
                rep_row,
                budget_rows,
                dimension_rows
            ) = run_replication(
                regime,
                r_idx,
                rep
            )

            STATE[
                "replication_rows"
            ].append(
                rep_row
            )

            STATE[
                "budget_rows"
            ].extend(
                budget_rows
            )

            STATE[
                "dimension_rows"
            ].extend(
                dimension_rows
            )

            STATE[
                "completed"
            ].add(
                task
            )

            save_checkpoint(
                force=True
            )

            print(
                "  B_hat =",
                rep_row[
                    "selected_budget"
                ],
                "| test MSE =",
                round(
                    rep_row[
                        "adaptive_test_mse"
                    ],
                    5
                ),
                "| common-d MSE =",
                round(
                    rep_row[
                        "common_dimension_test_mse"
                    ],
                    5
                ),
                "| refits =",
                rep_row[
                    "number_unique_refits"
                ],
                "/",
                rep_row[
                    "total_allocation_space"
                ]
            )

except KeyboardInterrupt:

    print(
        "\nInterrupted. Saving checkpoint..."
    )

    save_checkpoint(
        force=True
    )

    raise

except Exception:

    print(
        "\nError encountered. Saving checkpoint..."
    )

    save_checkpoint(
        force=True
    )

    raise


# ---------------------------
# 24. SAVE RESULTS
# ---------------------------

rep_df = pd.DataFrame(
    STATE[
        "replication_rows"
    ]
)

budget_df = pd.DataFrame(
    STATE[
        "budget_rows"
    ]
)

dimension_df = pd.DataFrame(
    STATE[
        "dimension_rows"
    ]
)

rep_df.to_csv(
    RESULTS_PATH,
    index=False
)

budget_df.to_csv(
    BUDGET_RESULTS_PATH,
    index=False
)

dimension_df.to_csv(
    DIMENSION_PATH,
    index=False
)

summary = (
    rep_df
    .groupby(
        "regime"
    )
    .agg(
        n_rep=(
            "rep",
            "count"
        ),
        adaptive_test_mse=(
            "adaptive_test_mse",
            "mean"
        ),
        common_dimension_test_mse=(
            "common_dimension_test_mse",
            "mean"
        ),
        max_budget_test_mse=(
            "max_budget_test_mse",
            "mean"
        ),
        reference_full_test_mse=(
            "reference_full_test_mse",
            "mean"
        ),
        adaptive_minus_common=(
            "adaptive_vs_common_test_difference",
            "mean"
        ),
        adaptive_minus_max_budget=(
            "adaptive_vs_max_budget_test_difference",
            "mean"
        ),
        mean_selected_budget=(
            "selected_budget",
            "mean"
        ),
        embedding_reduction=(
            "embedding_parameter_reduction",
            "mean"
        ),
        structural_excess=(
            "true_structural_excess_at_selected_budget",
            "mean"
        ),
        dimension_complexity_correlation=(
            "dimension_complexity_spearman",
            "mean"
        ),
        mean_unique_refits=(
            "number_unique_refits",
            "mean"
        ),
        refit_reduction_percent=(
            "refit_reduction_percent",
            "mean"
        ),
        dp_runtime_seconds=(
            "dp_runtime_seconds",
            "mean"
        ),
        exact_dp_check_rate=(
            "dp_exact_on_prefix",
            "mean"
        ),
        prefix_dp_speedup=(
            "prefix_dp_speedup",
            "mean"
        ),
        runtime_seconds=(
            "runtime_seconds",
            "mean"
        ),
    )
    .reset_index()
)

summary.to_csv(
    SUMMARY_PATH,
    index=False
)

# ---------------------------
# 25. FIGURES
# ---------------------------

x = np.arange(
    len(summary)
)

width = 0.25

plt.figure(
    figsize=(9, 5)
)

plt.bar(
    x - width,
    summary[
        "adaptive_test_mse"
    ],
    width,
    label="Adaptive"
)

plt.bar(
    x,
    summary[
        "common_dimension_test_mse"
    ],
    width,
    label="Common dimension"
)

plt.bar(
    x + width,
    summary[
        "max_budget_test_mse"
    ],
    width,
    label="Maximum budget"
)

plt.xticks(
    x,
    summary["regime"]
)

plt.ylabel(
    "Test MSE"
)

plt.legend()
plt.tight_layout()

plt.savefig(
    os.path.join(
        ROOT,
        "sim2_predictive_comparison.png"
    ),
    dpi=180
)

plt.close()


plt.figure(
    figsize=(8, 5)
)

plt.bar(
    summary[
        "regime"
    ],
    summary[
        "refit_reduction_percent"
    ]
)

plt.ylabel(
    "Candidate refits eliminated (%)"
)

plt.tight_layout()

plt.savefig(
    os.path.join(
        ROOT,
        "sim2_computational_reduction.png"
    ),
    dpi=180
)

plt.close()


# Mean selected dimension by predictor.
mean_dims = (
    dimension_df
    .groupby(
        [
            "regime",
            "predictor"
        ]
    )[
        [
            "selected_dimension",
            "true_effective_dimension_90"
        ]
    ]
    .mean()
    .reset_index()
)

for regime in REGIMES:

    tmp = mean_dims[
        mean_dims[
            "regime"
        ]
        == regime
    ]

    plt.figure(
        figsize=(10, 5)
    )

    plt.plot(
        tmp[
            "predictor"
        ],
        tmp[
            "selected_dimension"
        ],
        marker="o",
        label="Selected dimension"
    )

    plt.plot(
        tmp[
            "predictor"
        ],
        tmp[
            "true_effective_dimension_90"
        ],
        marker="s",
        label="True 90% spectral dimension"
    )

    plt.xlabel(
        "Categorical predictor"
    )

    plt.ylabel(
        "Dimension"
    )

    plt.xticks(
        tmp[
            "predictor"
        ]
    )

    plt.legend()
    plt.tight_layout()

    plt.savefig(
        os.path.join(
            ROOT,
            f"sim2_dimensions_{regime}.png"
        ),
        dpi=180
    )

    plt.close()


# ---------------------------
# 26. FINAL CHECKPOINT
# ---------------------------

STATE[
    "finished"
] = True

save_checkpoint(
    force=True
)

print(
    "\nSIMULATION 2 COMPLETED."
)

print(
    "\nSummary:"
)

print(
    summary.to_string(
        index=False
    )
)

print(
    "\nFiles saved in:"
)

print(ROOT)

Mounted at /content/drive

Simulation 2
---------------------------------------
Number of categorical predictors: 12
N_j: [3, 5, 8, 4, 10, 6, 7, 12, 5, 9, 4, 11]
K_j: [2, 4, 5, 3, 5, 5, 5, 5, 4, 5, 3, 5]
Candidate budgets: [84, 134, 184, 234, 285, 335, 385]
Total possible allocations: 22500000
Pilot mode: True
---------------------------------------


[heterogeneous] replication 1/6
[checkpoint] 10:07:27
  B_hat = 84 | test MSE = 0.09766 | common-d MSE = 0.09628 | refits = 7 / 22500000

[heterogeneous] replication 2/6
[checkpoint] 10:08:13
  B_hat = 84 | test MSE = 0.09935 | common-d MSE = 0.08224 | refits = 7 / 22500000

[heterogeneous] replication 3/6
[checkpoint] 10:09:00
  B_hat = 84 | test MSE = 0.10252 | common-d MSE = 0.08846 | refits = 7 / 22500000

[heterogeneous] replication 4/6
[checkpoint] 10:09:45
  B_hat = 84 | test MSE = 0.09945 | common-d MSE = 0.08901 | refits = 7 / 22500000

[heterogeneous] replication 5/6
[checkpoint] 10:10:31
  B_hat = 84 | test MSE = 0.111 | common